# Reconciliation: ledger.csv vs gateway.csv

This notebook performs a reconciliation between `01_data/raw/ledger.csv` and `01_data/raw/gateway.csv` and writes reconciliation outputs to `01_data/processed/` and a summary JSON to `04_python/summary_metrics.json`.

Steps:
- Load both files
- Check duplicates and nulls
- Identify records missing in gateway
- Identify records missing in ledger
- Identify amount mismatches
- Identify status mismatches
- Build final reconciliation report
- Generate summary metrics

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

# Locate project root by looking for the `01_data` folder in parents
cwd = Path.cwd()
project_root = None
for p in [cwd] + list(cwd.parents):
    if (p / '01_data').exists():
        project_root = p
        break
if project_root is None:
    project_root = cwd

raw_dir = project_root / '01_data' / 'raw'
processed_dir = project_root / '01_data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

ledger_fp = raw_dir / 'ledger.csv'
gateway_fp = raw_dir / 'gateway.csv'
summary_json_fp = project_root / '04_python' / 'summary_metrics.json'

print('Project root:', project_root)
print('Ledger file:', ledger_fp)
print('Gateway file:', gateway_fp)
print('Processed dir:', processed_dir)

Project root: d:\Python\BitSom\quickpay_data_analysis
Ledger file: d:\Python\BitSom\quickpay_data_analysis\01_data\raw\ledger.csv
Gateway file: d:\Python\BitSom\quickpay_data_analysis\01_data\raw\gateway.csv
Processed dir: d:\Python\BitSom\quickpay_data_analysis\01_data\processed


In [2]:
# Load CSVs
ledger = pd.read_csv(ledger_fp, parse_dates=['transaction_date'])
gateway = pd.read_csv(gateway_fp, parse_dates=['transaction_date'])

# Normalize basic types
for df in (ledger, gateway):
    df['transaction_id'] = df['transaction_id'].astype(str)
    df['merchant_id'] = df['merchant_id'].astype(str)
    df['amount_usd'] = pd.to_numeric(df['amount_usd'], errors='coerce')
    df['status'] = df['status'].astype(str)

print('Ledger rows:', len(ledger))
print('Gateway rows:', len(gateway))
print('Ledger head:')
print(ledger.head(5))
print('Gateway head:')
print(gateway.head(5))

Ledger rows: 10
Gateway rows: 9
Ledger head:
  transaction_id transaction_date merchant_id  amount_usd   status  \
0           R001       2026-03-01        M001      1200.0  success   
1           R002       2026-03-01        M002       850.0  success   
2           R003       2026-03-02        M001       500.0  success   
3           R004       2026-03-02        M003      2100.0  success   
4           R005       2026-03-03        M004      7200.0  success   

  payment_method  
0            UPI  
1           Card  
2         Wallet  
3           Card  
4           Card  
Gateway head:
  transaction_id transaction_date merchant_id  amount_usd   status  \
0           R001       2026-03-01        M001      1200.0  success   
1           R002       2026-03-01        M002       900.0  success   
2           R003       2026-03-02        M001       500.0  success   
3           R005       2026-03-03        M004      7200.0   failed   
4           R006       2026-03-03        M002       950.

In [3]:
# Check duplicates and nulls
dupes_ledger = ledger[ledger.duplicated(subset=['transaction_id'], keep=False)]
dupes_gateway = gateway[gateway.duplicated(subset=['transaction_id'], keep=False)]

print('Ledger duplicate count:', len(dupes_ledger))
print('Gateway duplicate count:', len(dupes_gateway))

print('Ledger nulls:', ledger.isnull().sum())
print('Gateway nulls:', gateway.isnull().sum())

Ledger duplicate count: 0
Gateway duplicate count: 0
Ledger nulls: transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64
Gateway nulls: transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64


In [4]:
# Merge to detect presence and mismatches
merged = ledger.merge(gateway, on='transaction_id', how='outer', suffixes=('_ledger', '_gateway'), indicator=True)

# Missing in gateway -> present in ledger but not in gateway
missing_in_gateway = merged[merged['_merge'] == 'left_only'].copy()
missing_in_gateway = missing_in_gateway[[
    'transaction_id',
    'transaction_date_ledger',
    'merchant_id_ledger',
    'amount_usd_ledger',
    'status_ledger',
    'payment_method_ledger'
]]
missing_in_gateway.columns = ['transaction_id','transaction_date','merchant_id','amount_usd','status','payment_method']
missing_in_gateway.to_csv(processed_dir / 'missing_in_gateway.csv', index=False)
print('Missing in gateway count:', len(missing_in_gateway))

# Missing in ledger -> present in gateway but not in ledger
missing_in_ledger = merged[merged['_merge'] == 'right_only'].copy()
missing_in_ledger = missing_in_ledger[[
    'transaction_id',
    'transaction_date_gateway',
    'merchant_id_gateway',
    'amount_usd_gateway',
    'status_gateway',
    'payment_method_gateway'
]]
missing_in_ledger.columns = ['transaction_id','transaction_date','merchant_id','amount_usd','status','payment_method']
missing_in_ledger.to_csv(processed_dir / 'missing_in_ledger.csv', index=False)
print('Missing in ledger count:', len(missing_in_ledger))

Missing in gateway count: 2
Missing in ledger count: 1


In [5]:
# Amount mismatches for transactions present in both
both = merged[merged['_merge'] == 'both'].copy()

def _is_amount_mismatch(a, b):
    if pd.isna(a) and pd.isna(b):
        return False
    if pd.isna(a) or pd.isna(b):
        return True
    return not np.isclose(a, b)

both['amount_mismatch'] = both.apply(lambda r: _is_amount_mismatch(r['amount_usd_ledger'], r['amount_usd_gateway']), axis=1)
amount_mismatches = both[both['amount_mismatch']].copy()
# Normalize columns and always write a CSV (may be empty)
amount_mismatches_out = amount_mismatches[[
    'transaction_id',
    'transaction_date_ledger',
    'merchant_id_ledger',
    'amount_usd_ledger',
    'amount_usd_gateway',
    'status_ledger',
    'status_gateway'
]].copy()
amount_mismatches_out.columns = ['transaction_id','transaction_date','merchant_id','amount_usd_ledger','amount_usd_gateway','status_ledger','status_gateway']
if not amount_mismatches_out.empty:
    amount_mismatches_out['amount_diff'] = amount_mismatches_out['amount_usd_ledger'] - amount_mismatches_out['amount_usd_gateway']
amount_mismatches_out.to_csv(processed_dir / 'amount_mismatches.csv', index=False)
print('Amount mismatches count:', len(amount_mismatches_out))

Amount mismatches count: 2


In [6]:
# Status mismatches for transactions present in both
both['status_mismatch'] = both['status_ledger'].fillna('').str.lower() != both['status_gateway'].fillna('').str.lower()
status_mismatches = both[both['status_mismatch']].copy()
status_mismatches_out = status_mismatches[[
    'transaction_id',
    'transaction_date_ledger',
    'merchant_id_ledger',
    'status_ledger',
    'status_gateway'
]].copy()
status_mismatches_out.columns = ['transaction_id','transaction_date','merchant_id','status_ledger','status_gateway']
status_mismatches_out.to_csv(processed_dir / 'status_mismatches.csv', index=False)
print('Status mismatches count:', len(status_mismatches_out))

Status mismatches count: 1


In [7]:
# Build final reconciliation report
report = merged.copy()
report['transaction_date'] = report['transaction_date_ledger'].combine_first(report['transaction_date_gateway'])
report['merchant_id'] = report['merchant_id_ledger'].combine_first(report['merchant_id_gateway'])
report['amount_usd_ledger'] = report['amount_usd_ledger']
report['amount_usd_gateway'] = report['amount_usd_gateway']
report['status_ledger'] = report['status_ledger']
report['status_gateway'] = report['status_gateway']

def _amount_match(a, b):
    if pd.isna(a) and pd.isna(b):
        return True
    if pd.isna(a) or pd.isna(b):
        return False
    return np.isclose(a, b)

report['amount_match'] = report.apply(lambda r: _amount_match(r['amount_usd_ledger'], r['amount_usd_gateway']), axis=1)
report['status_match'] = report['status_ledger'].fillna('').str.lower() == report['status_gateway'].fillna('').str.lower()
report['presence'] = report['_merge'].map({'both':'both','left_only':'missing_in_gateway','right_only':'missing_in_ledger'})

def _recon_status(r):
    if r['presence'] != 'both':
        return r['presence']
    if not r['amount_match']:
        return 'amount_mismatch'
    if not r['status_match']:
        return 'status_mismatch'
    return 'matched'

report['recon_status'] = report.apply(_recon_status, axis=1)
report_out = report[[
    'transaction_id', 'transaction_date', 'merchant_id',
    'amount_usd_ledger', 'amount_usd_gateway', 'amount_match',
    'status_ledger', 'status_gateway', 'status_match',
    'presence', 'recon_status'
]].copy()
report_out.to_csv(processed_dir / 'reconciliation_report.csv', index=False)
print('Wrote reconciliation_report.csv with', len(report_out), 'rows')

Wrote reconciliation_report.csv with 11 rows


In [8]:
# Summary metrics and write JSON
def _sum_amount_df(df):
    if df is None or df.empty:
        return 0.0
    for col in ['amount_usd', 'amount', 'amount_usd_ledger', 'amount_usd_gateway']:
        if col in df.columns:
            return float(pd.to_numeric(df[col], errors='coerce').sum(skipna=True))
    numcols = df.select_dtypes(include=['number']).columns.tolist()
    if numcols:
        return float(df[numcols[0]].sum(skipna=True))
    return 0.0

def _count_unique_tx(df):
    if df is None:
        return 0
    if 'transaction_id' in df.columns:
        return int(df['transaction_id'].nunique())
    return int(len(df))

total_ledger_rows = _count_unique_tx(ledger) if 'ledger' in locals() else 0
total_gateway_rows = _count_unique_tx(gateway) if 'gateway' in locals() else 0

missing_in_gateway_count = _count_unique_tx(missing_in_gateway) if 'missing_in_gateway' in locals() else 0
missing_in_ledger_count = _count_unique_tx(missing_in_ledger) if 'missing_in_ledger' in locals() else 0
amount_mismatch_count = _count_unique_tx(amount_mismatches_out) if 'amount_mismatches_out' in locals() else 0
status_mismatch_count = _count_unique_tx(status_mismatches_out) if 'status_mismatches_out' in locals() else 0

if 'report_out' in locals() and 'recon_status' in report_out.columns:
    reconciliation_issue_count = int((report_out['recon_status'] != 'matched').sum())
else:
    reconciliation_issue_count = missing_in_gateway_count + missing_in_ledger_count + amount_mismatch_count + status_mismatch_count

ledger_total_amount = _sum_amount_df(ledger) if 'ledger' in locals() else 0.0
gateway_total_amount = _sum_amount_df(gateway) if 'gateway' in locals() else 0.0

amount_at_risk = 0.0
if 'amount_mismatches_out' in locals() and not amount_mismatches_out.empty:
    if 'amount_diff' in amount_mismatches_out.columns:
        amount_at_risk += float(pd.to_numeric(amount_mismatches_out['amount_diff'], errors='coerce').abs().sum(skipna=True))
    else:
        if 'amount_usd_ledger' in amount_mismatches_out.columns and 'amount_usd_gateway' in amount_mismatches_out.columns:
            a = pd.to_numeric(amount_mismatches_out['amount_usd_ledger'], errors='coerce')
            b = pd.to_numeric(amount_mismatches_out['amount_usd_gateway'], errors='coerce')
            amount_at_risk += float((a - b).abs().sum(skipna=True))

for df in (missing_in_gateway if 'missing_in_gateway' in locals() else None, missing_in_ledger if 'missing_in_ledger' in locals() else None):
    if df is not None and not df.empty:
        for col in df.columns:
            if 'amount' in col.lower():
                amount_at_risk += float(pd.to_numeric(df[col], errors='coerce').abs().sum(skipna=True))
                break

ledger_total_amount = round(ledger_total_amount, 2)
gateway_total_amount = round(gateway_total_amount, 2)
amount_at_risk = round(amount_at_risk, 2)

metrics = {
    'total_ledger_rows': int(total_ledger_rows),
    'total_gateway_rows': int(total_gateway_rows),
    'missing_in_gateway_count': int(missing_in_gateway_count),
    'missing_in_ledger_count': int(missing_in_ledger_count),
    'amount_mismatch_count': int(amount_mismatch_count),
    'status_mismatch_count': int(status_mismatch_count),
    'reconciliation_issue_count': int(reconciliation_issue_count),
    'ledger_total_amount': ledger_total_amount,
    'gateway_total_amount': gateway_total_amount,
    'amount_at_risk': amount_at_risk
}
summary_json_fp.parent.mkdir(parents=True, exist_ok=True)
with open(summary_json_fp, 'w') as fh:
    json.dump(metrics, fh, indent=2)

print('Summary metrics:')
print(json.dumps(metrics, indent=2))

Summary metrics:
{
  "total_ledger_rows": 10,
  "total_gateway_rows": 9,
  "missing_in_gateway_count": 2,
  "missing_in_ledger_count": 1,
  "amount_mismatch_count": 2,
  "status_mismatch_count": 1,
  "reconciliation_issue_count": 6,
  "ledger_total_amount": 23340.0,
  "gateway_total_amount": 20550.0,
  "amount_at_risk": 6490.0
}


**Outputs written**:
- `01_data/processed/missing_in_gateway.csv`
- `01_data/processed/missing_in_ledger.csv`
- `01_data/processed/amount_mismatches.csv`
- `01_data/processed/status_mismatches.csv`
- `01_data/processed/reconciliation_report.csv`
- `04_python/summary_metrics.json`


## Part 4: JSON Normalization

Read `01_data/raw/api_response_sample.json`, flatten nested records into rows (one settlement per row), clean column names, convert datetime fields, and save to `01_data/processed/api_normalized.csv`.

In [11]:
import json
from pathlib import Path
import pandas as pd

# Paths
raw_json_fp = raw_dir / 'api_response_sample.json'
out_fp = processed_dir / 'api_normalized.csv'

# Load JSON
with open(raw_json_fp, 'r', encoding='utf-8') as fh:
    api = json.load(fh)

# Build records: one row per settlement with batch and merchant metadata
records = []
for batch in api.get('batches', []):
    batch_id = batch.get('batch_id')
    merchant = batch.get('merchant', {}) or {}
    for s in batch.get('settlements', []):
        rec = {}
        rec['batch_id'] = batch_id
        # merchant fields
        rec['merchant_id'] = merchant.get('merchant_id')
        rec['merchant_name'] = merchant.get('merchant_name')
        rec['merchant_region'] = merchant.get('region')
        # settlement top-level fields
        rec['settlement_id'] = s.get('settlement_id')
        rec['amount_usd'] = s.get('amount_usd')
        rec['status'] = s.get('status')
        rec['processed_at'] = s.get('processed_at')
        # bank nested
        bank = s.get('bank') or {}
        rec['bank_name'] = bank.get('name')
        rec['bank_country'] = bank.get('country')
        records.append(rec)

# Add top-level generated_at and source to each row
generated_at = api.get('generated_at')
source = api.get('source')
for r in records:
    r['generated_at'] = generated_at
    r['source'] = source

df_api = pd.DataFrame.from_records(records)
print('Rows flattened:', len(df_api))
df_api.head()

Rows flattened: 6


,batch_id,merchant_id,merchant_name,merchant_region,settlement_id,amount_usd,status,processed_at,bank_name,bank_country,generated_at,source
0,B001,M001,Alpha Mart,APAC,S001,1520.5,settled,2026-03-07T08:10:00Z,Bank A,IN,2026-03-07T10:00:00Z,QuickPay Settlement API
1,B001,M001,Alpha Mart,APAC,S002,980.0,pending,2026-03-07T08:45:00Z,Bank A,IN,2026-03-07T10:00:00Z,QuickPay Settlement API
2,B001,M001,Alpha Mart,APAC,S003,640.0,settled,2026-03-07T09:15:00Z,Bank B,SG,2026-03-07T10:00:00Z,QuickPay Settlement API
3,B002,M004,Delta Travels,US,S004,2100.0,settled,2026-03-07T08:20:00Z,Bank C,US,2026-03-07T10:00:00Z,QuickPay Settlement API
4,B002,M004,Delta Travels,US,S005,500.0,failed,2026-03-07T08:50:00Z,Bank C,US,2026-03-07T10:00:00Z,QuickPay Settlement API


In [12]:
# Clean column names
df = df_api.copy()
df.columns = (df.columns.str.strip()
                .str.lower()
                .str.replace(' ', '_')
                .str.replace(r'[^0-9a-z_]', '', regex=True))

# Convert types
df['amount_usd'] = pd.to_numeric(df['amount_usd'], errors='coerce')
df['processed_at'] = pd.to_datetime(df['processed_at'], utc=True, errors='coerce')
df['generated_at'] = pd.to_datetime(df['generated_at'], utc=True, errors='coerce')

# Reorder columns to a sensible layout if present
cols_order = ['generated_at','source','batch_id','settlement_id','merchant_id','merchant_name','merchant_region','amount_usd','status','processed_at','bank_name','bank_country']
cols = [c for c in cols_order if c in df.columns] + [c for c in df.columns if c not in cols_order]
df = df[cols]

# Save normalized CSV
out_fp.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_fp, index=False)
print('Wrote', out_fp, 'with', len(df), 'rows')
df.head()

Wrote d:\Python\BitSom\quickpay_data_analysis\01_data\processed\api_normalized.csv with 6 rows


,generated_at,source,batch_id,settlement_id,merchant_id,merchant_name,merchant_region,amount_usd,status,processed_at,bank_name,bank_country
0,2026-03-07 10:00:00+00:00,QuickPay Settlement API,B001,S001,M001,Alpha Mart,APAC,1520.5,settled,2026-03-07 08:10:00+00:00,Bank A,IN
1,2026-03-07 10:00:00+00:00,QuickPay Settlement API,B001,S002,M001,Alpha Mart,APAC,980.0,pending,2026-03-07 08:45:00+00:00,Bank A,IN
2,2026-03-07 10:00:00+00:00,QuickPay Settlement API,B001,S003,M001,Alpha Mart,APAC,640.0,settled,2026-03-07 09:15:00+00:00,Bank B,SG
3,2026-03-07 10:00:00+00:00,QuickPay Settlement API,B002,S004,M004,Delta Travels,US,2100.0,settled,2026-03-07 08:20:00+00:00,Bank C,US
4,2026-03-07 10:00:00+00:00,QuickPay Settlement API,B002,S005,M004,Delta Travels,US,500.0,failed,2026-03-07 08:50:00+00:00,Bank C,US


## Part 5: Generate risk-derived reports

Read `01_data/processed/merchant_risk_summary.csv` and produce the following reports in `01_data/processed/`:
- `daily_summary.csv`
- `payment_method_breakdown.csv`
- `region_breakdown.csv`
- `merchant_performance_summary.csv`

In [9]:
# Load and preprocess merchant risk summary
mr_fp = processed_dir / 'merchant_risk_summary.csv'
print('Reading', mr_fp)
df_mr = pd.read_csv(mr_fp)
# Normalize numeric fields
df_mr['amount_usd'] = pd.to_numeric(df_mr.get('amount_usd', pd.Series()), errors='coerce')
df_mr['risk_score'] = pd.to_numeric(df_mr.get('risk_score', pd.Series()), errors='coerce')
df_mr['high_risk_flag'] = pd.to_numeric(df_mr.get('high_risk_flag', pd.Series()), errors='coerce').fillna(0).astype(int)
df_mr['high_value_flag'] = pd.to_numeric(df_mr.get('high_value_flag', pd.Series()), errors='coerce').fillna(0).astype(int)
# Parse transaction date (DD-MM-YYYY in source)
if 'transaction_date' in df_mr.columns:
    df_mr['transaction_date_parsed'] = pd.to_datetime(df_mr['transaction_date'], format='%d-%m-%Y', errors='coerce')
    df_mr['date'] = df_mr['transaction_date_parsed'].dt.strftime('%Y-%m-%d')
else:
    df_mr['date'] = None
# Derive status flags
status = df_mr.get('status', pd.Series()).fillna('').astype(str).str.upper()
df_mr['is_chargeback'] = status.str.contains('CHARGEBACK', na=False).astype(int)
df_mr['is_failed'] = status.str.contains('FAILED', na=False).astype(int)
df_mr['is_captured'] = (status == 'CAPTURED').astype(int)
print('Loaded rows:', len(df_mr))
df_mr.head()

Reading d:\Python\BitSom\quickpay_data_analysis\01_data\processed\merchant_risk_summary.csv
Loaded rows: 30


,transaction_id,transaction_date,merchant_id,merchant_name,merchant_account_manager,merchant_category,currency,amount_usd,high_value_flag,status,risk_score,high_risk_flag,gateway_region,user_id,payment_method,transaction_date_parsed,date,is_chargeback,is_failed,is_captured
0,T001,01-03-2026,M001,ALPHA MART,Aisha Khan,Grocery,INR,4998.0,0,CAPTURED,62.0,0,APAC,U001,UPI,2026-03-01,2026-03-01,0,0,1
1,T002,01-03-2026,M001,ALPHA MART,Aisha Khan,Grocery,INR,2499.0,0,CAPTURED,55.0,0,MISSING,U002,Card,2026-03-01,2026-03-01,0,0,1
2,T003,01-03-2026,M002,BETA STORES,Rohan Mehta,Electronics,INR,6069.0,1,CAPTURED,71.0,1,APAC,U003,NetBanking,2026-03-01,2026-03-01,0,0,1
3,T004,02-03-2026,M002,BETA STORES,Rohan Mehta,Electronics,INR,1920.0,0,FAILED E05 TIMEOUT,68.0,0,APAC,U004,Card,2026-03-02,2026-03-02,0,1,0
4,T005,02-03-2026,M001,ALPHA MART,Aisha Khan,Grocery,INR,4680.0,0,CAPTURED,58.0,0,MISSING,U001,UPI,2026-03-02,2026-03-02,0,0,1


In [10]:
# Aggregations and write outputs
out_dir = processed_dir
# 1) daily_summary.csv
daily = df_mr.groupby('date', dropna=False).agg(
    transactions_count=('transaction_id','count'),
    total_amount_usd=('amount_usd','sum'),
    avg_risk_score=('risk_score','mean'),
    high_risk_count=('high_risk_flag','sum'),
    high_value_count=('high_value_flag','sum'),
    chargeback_count=('is_chargeback','sum'),
    failed_count=('is_failed','sum'),
    captured_count=('is_captured','sum'),
).reset_index().sort_values('date')
daily.to_csv(out_dir / 'daily_summary.csv', index=False)
print('Wrote', out_dir / 'daily_summary.csv')
display(daily.head())
# 2) payment_method_breakdown.csv
pm = df_mr.groupby('payment_method', dropna=False).agg(
    transactions_count=('transaction_id','count'),
    total_amount_usd=('amount_usd','sum'),
    avg_risk_score=('risk_score','mean'),
    high_risk_count=('high_risk_flag','sum'),
    chargeback_count=('is_chargeback','sum'),
).reset_index().sort_values('total_amount_usd', ascending=False)
pm.to_csv(out_dir / 'payment_method_breakdown.csv', index=False)
print('Wrote', out_dir / 'payment_method_breakdown.csv')
display(pm.head())
# 3) region_breakdown.csv
region = df_mr.groupby('gateway_region', dropna=False).agg(
    transactions_count=('transaction_id','count'),
    total_amount_usd=('amount_usd','sum'),
    avg_risk_score=('risk_score','mean'),
    high_risk_count=('high_risk_flag','sum'),
    chargeback_count=('is_chargeback','sum'),
).reset_index().sort_values('total_amount_usd', ascending=False)
region.to_csv(out_dir / 'region_breakdown.csv', index=False)
print('Wrote', out_dir / 'region_breakdown.csv')
display(region.head())
# 4) merchant_performance_summary.csv
merchant = df_mr.groupby(['merchant_id','merchant_name'], dropna=False).agg(
    transactions_count=('transaction_id','count'),
    total_amount_usd=('amount_usd','sum'),
    avg_risk_score=('risk_score','mean'),
    chargeback_count=('is_chargeback','sum'),
    high_risk_count=('high_risk_flag','sum'),
    high_value_count=('high_value_flag','sum'),
).reset_index()
merchant['chargeback_rate'] = (merchant['chargeback_count'] / merchant['transactions_count']).fillna(0)
merchant = merchant.sort_values('total_amount_usd', ascending=False)
merchant.to_csv(out_dir / 'merchant_performance_summary.csv', index=False)
print('Wrote', out_dir / 'merchant_performance_summary.csv')
display(merchant.head())

Wrote d:\Python\BitSom\quickpay_data_analysis\01_data\processed\daily_summary.csv


,date,transactions_count,total_amount_usd,avg_risk_score,high_risk_count,high_value_count,chargeback_count,failed_count,captured_count
0,2026-03-01,5,26382.0,55.800000,1,2,0,0,5
1,2026-03-02,6,25049.0,63.166667,2,2,2,1,3
2,2026-03-03,5,18391.0,55.000000,1,0,0,1,4
3,2026-03-04,5,16420.0,59.600000,2,1,1,0,4
4,2026-03-05,6,19232.0,68.833333,3,1,1,4,1


Wrote d:\Python\BitSom\quickpay_data_analysis\01_data\processed\payment_method_breakdown.csv


,payment_method,transactions_count,total_amount_usd,avg_risk_score,high_risk_count,chargeback_count
0,Card,13,52972.0,61.307692,4,2
2,UPI,9,34397.5,62.625000,2,1
1,NetBanking,3,15306.0,62.666667,2,0
3,Wallet,5,13404.5,54.600000,1,1


Wrote d:\Python\BitSom\quickpay_data_analysis\01_data\processed\region_breakdown.csv


,gateway_region,transactions_count,total_amount_usd,avg_risk_score,high_risk_count,chargeback_count
0,APAC,13,50177.5,67.538462,6,2
2,MISSING,9,32416.5,62.125000,1,0
1,EU,4,18886.0,47.250000,1,1
3,US,4,14600.0,48.750000,1,1


Wrote d:\Python\BitSom\quickpay_data_analysis\01_data\processed\merchant_performance_summary.csv


,merchant_id,merchant_name,transactions_count,total_amount_usd,avg_risk_score,chargeback_count,high_risk_count,high_value_count,chargeback_rate
1,M002,BETA STORES,11,41782.0,69.363636,1,5,2,0.090909
0,M001,ALPHA MART,11,40812.0,61.200000,1,2,2,0.090909
3,M004,DELTA TRAVELS,4,14600.0,48.750000,1,1,1,0.250000
4,M005,ECO HOME,2,10246.0,54.500000,1,1,1,0.500000
2,M003,CITY PHARMA,2,8640.0,40.000000,0,0,0,0.000000
